To plot the reliability vs horizontal distance for each combination of USI, MCS, and Height.
To save these plots in a directory.

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

link = "Video" # "Downlink", "Uplink", "Video"

# Training Dataset
dataset_file_path = f"/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/data_processed/{link}_Reliability.csv"
save_path = f"/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/training_data_reliability_vs_hdist_plots/{link.lower()}/"

# Testing Dataset
# dataset_file_path = f"/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/complete_testing_dmax_dataset/data_processed_complete/{link}_Reliability.csv"
# save_path = f"/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/complete_test_data_reliability_vs_hdist_plots/{link.lower()}/"

data_df = pd.read_csv(dataset_file_path)

os.makedirs(save_path, exist_ok=True)

if "Num_Sent" in data_df.columns and "Num_Reliable" in data_df.columns:
    data_df["Reliability"] = data_df["Num_Reliable"] / data_df["Num_Sent"]
elif "Num_Fail_Other" in data_df.columns and "Num_Delay_Excd" in data_df.columns and "Num_Reliable" in data_df.columns:
    data_df["Reliability"] = data_df["Num_Reliable"] / (data_df["Num_Reliable"] + data_df["Num_Delay_Excd"] + data_df["Num_Fail_Other"])

for height in data_df['Height'].unique():
    for usi in data_df['UAV_Sending_Interval'].unique():
        for bitrate in data_df['Bitrate'].unique():
            subset = data_df[(data_df['Height'] == height) & (data_df['UAV_Sending_Interval'] == usi) & (data_df['Bitrate'] == bitrate)]
            subset = subset[subset["Horizontal_Distance"] <= 700]  # Limit to 700 meters
            plt.figure(figsize=(10, 4))
            plt.rcParams.update({'font.size': 14})
            plt.plot(subset["Horizontal_Distance"], subset["Reliability"], label="Reliability", color='blue')
            plt.ylabel("Reliability")
            plt.xlabel("Horizontal Distance (m)")
            plt.title(f"Reliability vs Horizontal Distance\nLink: {link}, Height: {height} m, USI: {usi} ms, Bitrate: {bitrate} kbps")
            plt.ylim(0, 1.05)
            # Save the plot with a descriptive filename
            filename = f"Reliability_Height-{height}_USI-{usi}_Bitrate-{bitrate}.png"
            plt.grid(True)
            plt.savefig(os.path.join(save_path, filename))
            plt.close()